# Davinci Notebook 14 — Inference, Generation, Tool Calls & Export

Generic autoregressive sampling helpers, export metadata, and a strict host-controlled tool-call protocol. The sampling wrapper remains a slow full-prefix baseline; the canonical model exposes prefill/decode primitives, but a production generation server, streaming, batching, and policy layer are still separate work.


In [ ]:
import torch, json, os
from pathlib import Path


## 1. Sampling functions

In [ ]:
def sample_next(logits,temperature=1.,top_k=None,top_p=1.):
 if temperature<=0: return logits.argmax(-1,keepdim=True)
 logits=logits/temperature; logits=filter_top_k(logits,top_k); logits=filter_top_p(logits,top_p)
 return torch.multinomial(torch.softmax(logits,-1),1)


## 2. Generic generation wrapper
The caller must supply a model with an implementation of `model(input_ids)` and a tokenizer with encode/decode methods. This wrapper is slow because it recomputes the full prefix each step.

In [ ]:
@torch.no_grad()
def generate_slow(model,input_ids,max_new_tokens,temperature=1.,top_k=None,top_p=1.,eos_id=None):
 model.eval(); ids=input_ids
 for _ in range(max_new_tokens):
  logits=model(ids)[:,-1,:]; nxt=sample_next(logits,temperature,top_k,top_p); ids=torch.cat((ids,nxt),dim=1)
  if eos_id is not None and (nxt==eos_id).all(): break
 return ids


## 3. Export metadata and reproducibility manifest

In [ ]:
def export_metadata(path,model_config,tokenizer_meta,revision,license_info):
 metadata={'model_config':model_config,'tokenizer':tokenizer_meta,'revision':revision,'license':license_info,'export_notes':'weights must be exported separately; record architecture and tokenizer revisions'}
 Path(path).write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8'); return metadata


## 4. Deployment gates
- Confirm tokenizer/model special-token IDs agree.
- Confirm checkpoint loads strictly with no missing/unexpected keys.
- Test cached generation parity and EOS handling.
- Validate JSON tool-call output against schema.
- Apply safety and permission checks before tool execution.
- Add stop sequences, repetition policies, and request limits.
- Measure latency, throughput, memory, and concurrent requests.
- Test quantized models against a reference set; do not infer quality from size reduction.
- Publish model card, license, data provenance, limitations, and evaluation results.


## 4. Strict host-controlled tool-call protocol
The host registers only approved functions. Model output is parsed as strict JSON and validated against the registered schema; generated Python or shell is never executed. The example uses a harmless arithmetic function. The host application must still own user authorization, consent, timeouts, rate limits, sandboxing, and audit logs.


In [ ]:
import sys, subprocess
REPO_DIR=Path('/content/Davinci')
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/keyhmoham/Davinci.git',str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
sys.path.insert(0,str(REPO_DIR))
from src.davinci.tool_call import ToolRegistry, ToolCallError
registry=ToolRegistry()
registry.register('add','Add two integers',{
    'type':'object','properties':{'left':{'type':'integer'},'right':{'type':'integer'}},
    'required':['left','right'],'additionalProperties':False,
},lambda left,right:{'sum':left+right})
model_facing_tools=registry.definitions()
model_output='{"name":"add","arguments":{"left":19,"right":23}}'
tool_result_json=registry.execute_json(model_output)
print('Registered tools:',json.dumps(model_facing_tools,ensure_ascii=False,indent=2))
print('Tool result:',tool_result_json)
try:
    registry.execute('{"name":"add","arguments":{"left":19,"right":23,"command":"rm -rf /"}}')
except ToolCallError as exc:
    print('Rejected unsafe/unexpected arguments:',exc)


## 5. Cached generation with the canonical decoder
This calls the repository's `generate()` helper, which uses one prompt prefill and incremental KV-cache decoding. It is a tiny smoke test; it does not validate throughput or quality at the target 500M scale.


In [ ]:
from src.davinci import DavinciConfig, DavinciForCausalLM, generate
cfg=DavinciConfig(vocab_size=257,hidden_size=64,num_hidden_layers=4,num_attention_heads=4,
    num_key_value_heads=2,head_dim=16,intermediate_size=128,local_window=8,
    global_every_n_layers=2,attention_chunk_size=4,global_attention_chunk_size=3)
model=DavinciForCausalLM(cfg).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
prompt=torch.randint(0,cfg.vocab_size,(1,12),device=next(model.parameters()).device)
generated=generate(model,prompt,max_new_tokens=8,temperature=0,top_k=None,top_p=1.0)
print({'prompt_tokens':prompt.shape[1],'generated_tokens':generated.shape[1]-prompt.shape[1],
       'token_ids':generated[0].tolist()})
